# Flow Reversal Steering による死角物体の探索・把持

π0.5（学習済み）を追加学習なしで使い、人間の大まかな方向の指示を FRS で π0.5 の動きに変換して誘導する。ランタイムのタイプで GPU（T4）を選び、上から順に実行する（アカウントや認証情報は不要）。

## セル0: セットアップ（ライブラリのインストール・環境変数・トークナイザの取得）

In [ ]:
# セル0: セットアップ（ライブラリのインストール・環境変数・トークナイザの取得）
import importlib
import importlib.util
import os
import pathlib
import subprocess
import sys

# MuJoCo を画面なしで描画する
os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
os.environ.setdefault("MUJOCO_EGL_DEVICE_ID", "0")

# Colab の GPU ランタイムは NVIDIA の EGL を glvnd に登録していないため、MUJOCO_GL=egl でも
# CPU のソフトウェア描画（Mesa llvmpipe）に落ちる。ドライバと同じ版の libEGL_nvidia は
# /usr/lib64-nvidia にあるので、それを登録して GPU 描画にする（版が違う libEGL_nvidia を apt で入れると
# EGL デバイスが 0 個になり描画できない。2026-10-24 に確認）。
# （プロセス内で最初に描画する前に実行する必要がある）
EGL_VENDOR = pathlib.Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
driver = subprocess.run(
    ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"], capture_output=True, text=True
).stdout.strip()
egl_lib = next(
    (str(p) for d in ["/usr/lib64-nvidia", "/usr/lib/x86_64-linux-gnu"] for p in pathlib.Path(d).glob(f"libEGL_nvidia.so.{driver}")),
    None,
)
if egl_lib:
    EGL_VENDOR.write_text('{"file_format_version": "1.0.0", "ICD": {"library_path": "%s"}}\n' % egl_lib)
print(f"NVIDIA driver {driver} / EGL: {egl_lib or '見つからない（CPU 描画のまま。動作はするが遅い）'}")

LEROBOT_REV = "d3793b7e36522321c1c455f0dc4826db3793dceb"  # LeRobot main (2026-10-23)
if importlib.util.find_spec("lerobot") is None:
    # LIBERO が依存する egl_probe は、pip の隔離ビルド環境の cmake ではビルドに失敗するため、
    # システムの cmake を使って先に入れておく（Colab, Python 3.13 で確認）
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "hf-egl-probe", "egl_probe"],
        check=True,
    )
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            f"lerobot[libero,pi] @ git+https://github.com/huggingface/lerobot.git@{LEROBOT_REV}",
            "imageio[ffmpeg]",
            # Colab 既定の torchao 0.10 では、LeRobot が読み込む diffusers の import が失敗する
            "torchao>=0.18",
        ],
        check=True,
    )
    importlib.invalidate_caches()

# LIBERO は初回 import 時に input() で保存先を尋ねてくるので、既定値の config.yaml を先に書いておく
libero_cfg = pathlib.Path(os.environ.get("LIBERO_CONFIG_PATH", "~/.libero")).expanduser() / "config.yaml"
if not libero_cfg.exists():
    import yaml

    root = pathlib.Path(importlib.util.find_spec("libero.libero").origin).parent
    libero_cfg.parent.mkdir(parents=True, exist_ok=True)
    libero_cfg.write_text(
        yaml.safe_dump(
            {
                "benchmark_root": str(root),
                "bddl_files": str(root / "bddl_files"),
                "init_states": str(root / "init_files"),
                "datasets": str(root.parent / "datasets"),
                "assets": str(root / "assets"),
            }
        )
    )

# π0.5 のトークナイザ（google/paligemma-3b-pt-224）は Hugging Face では gated（ログインとライセンス同意が必要）。
# 認証なしで実行できるよう、このリポジトリに同梱したもの（Gemma 利用規約に従い無改変で再配布）を取得して使う。
REPO_RAW = "https://raw.githubusercontent.com/pai-adv-public/pai-final-2026/main"
TOKENIZER_DIR = pathlib.Path("/content/paligemma_tokenizer")
TOKENIZER_FILES = ["tokenizer.json", "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json", "tokenizer.model"]
if not all((TOKENIZER_DIR / f).exists() for f in TOKENIZER_FILES):
    import urllib.request

    TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
    for f in TOKENIZER_FILES:
        urllib.request.urlretrieve(f"{REPO_RAW}/assets/paligemma_tokenizer/{f}", TOKENIZER_DIR / f)

OUT = pathlib.Path("/content/out")
OUT.mkdir(parents=True, exist_ok=True)
print("setup ok")

## セル1: π0.5（学習済み）と LIBERO シミュレータの読み込み

In [ ]:
# セル1: π0.5（学習済み）と LIBERO シミュレータの読み込み
import time

import numpy as np
import torch

from lerobot.configs.policies import PreTrainedConfig
from lerobot.envs.configs import LiberoEnv as LiberoEnvConfig
from lerobot.envs.factory import make_env, make_env_pre_post_processors
from lerobot.envs.utils import preprocess_observation
from lerobot.policies.factory import make_policy, make_pre_post_processors

POLICY_ID = "lerobot/pi05_libero_finetuned"
SUITE, TASK_ID = "libero_object", 0
N_ACTION_STEPS = 10  # 1 回の推論で実行するステップ数（openpi と同じ）
DEVICE = "cuda"

# --- シミュレータ（LeRobot の評価と同じ設定: 固定カメラ + 手先カメラ, 360x360）---
env_cfg = LiberoEnvConfig(task=SUITE, task_ids=[TASK_ID])
vec_env = make_env(env_cfg, n_envs=1)[SUITE][TASK_ID]  # SyncVectorEnv（バッチ次元 1）
env = vec_env.envs[0]  # lerobot.envs.libero.LiberoEnv
print("task:", env.task_description, "| max steps:", env._max_episode_steps)

# --- 方策 ---
t0 = time.time()
policy_cfg = PreTrainedConfig.from_pretrained(POLICY_ID)
policy_cfg.pretrained_path = POLICY_ID
policy_cfg.n_action_steps = N_ACTION_STEPS
policy_cfg.device = DEVICE
policy_cfg.compile_model = False  # Hub の設定では有効だが、初回の自動チューニングに約 8 分かかるため無効にする
assert not policy_cfg.use_relative_actions and not policy_cfg.use_visual_memory
# T4（メインメモリ 12 GB・VRAM 15 GB）でも読み込めるよう、モデルを最初から GPU 上に bf16 で作る。
# （LeRobot の既定では CPU 上に fp32 で作ってから bf16 に変換するため、約 15 GB のメインメモリが要る）
torch.set_default_dtype(torch.bfloat16)
try:
    with torch.device(DEVICE):
        policy = make_policy(cfg=policy_cfg, env_cfg=env_cfg)
finally:
    torch.set_default_dtype(torch.float32)
# 上の作り方だと、本来 fp32 の行動側の層（action_in_proj など）まで bf16 になるので、チェックポイントから fp32 で読み直す
from huggingface_hub import hf_hub_download
from safetensors import safe_open

_params = dict(policy.named_parameters())
with safe_open(hf_hub_download(POLICY_ID, "model.safetensors"), framework="pt") as f:
    for key in f.keys():
        if "paligemma_with_expert" not in key and key in _params:
            _params[key].data = f.get_tensor(key).to(DEVICE)
assert all(p.dtype == torch.float32 for k, p in _params.items() if "paligemma_with_expert" not in k)
policy.eval()
pre, post = make_pre_post_processors(
    policy_cfg=policy_cfg,
    pretrained_path=POLICY_ID,
    preprocessor_overrides={
        "device_processor": {"device": DEVICE},
        "rename_observations_processor": {"rename_map": {}},
        "tokenizer_processor": {"tokenizer_name": str(TOKENIZER_DIR)},  # 同梱のトークナイザ（認証不要）
    },
)
env_pre, _ = make_env_pre_post_processors(env_cfg=env_cfg, policy_cfg=policy_cfg)
print(
    f"policy loaded in {time.time() - t0:.0f}s | dtype={policy_cfg.dtype} "
    f"| chunk={policy_cfg.chunk_size} | steps={policy_cfg.num_inference_steps} "
    f"| GPU mem {torch.cuda.memory_allocated() / 2**30:.1f} GiB"
)


def to_batch(obs):
    """vec_env の観測を π0.5 の入力バッチに変換する（lerobot-eval と同じ順序）。"""
    batch = preprocess_observation(obs)
    batch["task"] = [env.task_description]
    batch = env_pre(batch)
    return pre(batch)


def frame_of(obs):
    """動画用: 固定カメラと手先カメラを横に並べる。
    robosuite の画像は上下が逆なので上下だけ戻す（π0.5 への入力は env_pre が学習時と同じ 180° 回転にする。表示とは別）。"""
    agent = obs["pixels"]["image"][0][::-1]
    wrist = obs["pixels"]["image2"][0][::-1]
    return np.concatenate([agent, wrist], axis=1)


@torch.no_grad()
def pi05_chunk(batch, noise=None):
    """通常の π0.5: 正規化された行動空間で (1, N_ACTION_STEPS, 7) を返す。"""
    return policy.predict_action_chunk(batch, noise=noise)[:, :N_ACTION_STEPS]


def run_episode(chunk_fn, init_state=0, seed=0, max_steps=None):
    """chunk_fn(batch, obs) -> 正規化行動チャンク を繰り返し実行する。"""
    max_steps = max_steps or env._max_episode_steps
    env.init_state_id = init_state
    obs, _ = vec_env.reset(seed=seed)
    frames, success, steps, infer_sec = [frame_of(obs)], False, 0, []
    while steps < max_steps and not success:
        batch = to_batch(obs)
        t = time.time()
        chunk = chunk_fn(batch, obs)
        infer_sec.append(time.time() - t)
        for i in range(chunk.shape[1]):
            action = post(chunk[:, i]).cpu().numpy()
            obs, _, terminated, _, _ = vec_env.step(action)
            frames.append(frame_of(obs))
            steps += 1
            success = bool(env._env.check_success())
            if success or terminated[0] or steps >= max_steps:
                break
    return {"success": success, "steps": steps, "frames": frames, "infer_sec": infer_sec}


def show_video(frames, name, fps=20):
    """フレーム列を MP4 に保存し、Notebook 内で再生する。"""
    import imageio.v2 as imageio
    from IPython.display import Video, display

    path = OUT / name
    imageio.mimsave(path, frames, fps=fps, macro_block_size=8)
    display(Video(str(path), embed=True, width=720))

## セル2: 元の配置で π0.5 に指示文だけを与える（スープ缶はカメラに映っている）

In [ ]:
# セル2: 元の配置で π0.5 に指示文だけを与える（スープ缶はカメラに映っている）
print("指示文:", env.task_description)
res = run_episode(lambda batch, obs: pi05_chunk(batch), init_state=0, seed=0)
show_video(res["frames"], "original.mp4")  # 左: 固定カメラ / 右: 手先カメラ
if res["success"]:
    print(f"成功: π0.5 がスープ缶を掴んで、かごに入れた（{res['steps']} ステップ）")
else:
    print(f"失敗: {res['steps']} ステップ以内にスープ缶をかごに入れられなかった")

## セル3: スープ缶をかごの裏に移した配置で、π0.5 に指示文だけを与える

In [ ]:
# セル3: スープ缶をかごの裏に移した配置で、π0.5 に指示文だけを与える
# 缶は固定カメラからはかごに隠れ、手先カメラにも映らない。人間は手伝わない。
import imageio.v2 as imageio
from IPython.display import Image, display

TARGET = "alphabet_soup_1"
HIDE_XY = (-0.15, 0.30)  # かごの裏（ロボット側）
IDLE = np.array([[0, 0, 0, 0, 0, 0, -1.0]])  # 何もしない行動（グリッパーは開）
MAX_STEPS = 400
NAMES = {  # 結果の表示用
    "alphabet_soup_1": "スープ缶（目標物）", "salad_dressing_1": "サラダドレッシングの瓶", "cream_cheese_1": "クリームチーズ",
    "milk_1": "牛乳パック", "tomato_sauce_1": "トマトソース", "butter_1": "バター",
}


def reset_hidden(seed=0):
    """元の初期状態から、スープ缶だけをかごの裏に移して落ち着かせる。"""
    env.init_state_id = 0
    vec_env.reset(seed=seed)
    sim = env._env.sim  # リセットのたびに作り直されるので毎回取り直す
    joint = env._env.env.objects_dict[TARGET].joints[0]
    qpos = sim.data.get_joint_qpos(joint).copy()
    qpos[:2] = HIDE_XY
    sim.data.set_joint_qpos(joint, qpos)
    sim.forward()
    for _ in range(10):
        obs, *_ = vec_env.step(IDLE)
    return obs


def objects_in_basket():
    """かごの中（かごの中心から水平 9 cm 以内）にある物体の名前。"""
    sim, objs = env._env.sim, env._env.env.objects_dict
    pos = {n: sim.data.body_xpos[sim.model.body_name2id(o.root_body)] for n, o in objs.items()}
    return [n for n in objs if n != "basket_1" and np.linalg.norm(pos[n][:2] - pos["basket_1"][:2]) < 0.09]


def run_pi05_hidden(seed=0):
    """缶を隠した配置で、π0.5 だけで最後まで動かす。"""
    torch.manual_seed(seed)
    obs = reset_hidden(seed)
    frames, steps, success = [frame_of(obs)], 0, False
    while steps < MAX_STEPS and not success:
        chunk = pi05_chunk(to_batch(obs))
        for i in range(chunk.shape[1]):
            obs, _, terminated, _, _ = vec_env.step(post(chunk[:, i]).cpu().numpy())
            frames.append(frame_of(obs))
            steps += 1
            success = bool(env._env.check_success())
            if success or terminated[0] or steps >= MAX_STEPS:
                break
    return {"success": success, "steps": steps, "frames": frames}


obs = reset_hidden()
imageio.imwrite(OUT / "scene_hidden.png", frame_of(obs))
display(Image(str(OUT / "scene_hidden.png")))  # 開始時: どちらのカメラにもスープ缶は映っていない

res_pi05 = run_pi05_hidden()
show_video(res_pi05["frames"], "hidden_pi05_only.mp4")
in_basket = [NAMES.get(n, n) for n in objects_in_basket()]
if res_pi05["success"]:
    print("成功: π0.5 がスープ缶をかごに入れた")
elif in_basket:
    print(f"失敗: 缶が見えないため、別の物体（{'、'.join(in_basket)}）をかごに入れてしまった")
else:
    print("失敗: 缶が見えないため、スープ缶をかごに入れられなかった")

## セル4: Flow Reversal Steering (FRS) の実装

In [ ]:
# セル4: Flow Reversal Steering (FRS) の実装
# 参考: Tang et al., "Improving Robotic Generalist Policies via Flow Reversal Steering" (arXiv 2606.13675)
#   1. 粗い参照行動 a_ref（例: 「左へ動け」）を作る
#   2. 観測で条件づけた流れの ODE を逆向きに積分し、a_ref に対応するノイズ â0 を求める
#   3. â0 から通常どおり順向きに積分すると、a_ref に近い「π0.5 らしい」行動が得られる
# π0.5 (openpi / LeRobot) は t=1 がノイズ、t=0 が行動の規約（NOISE_AT_ONE）。
from lerobot.policies.common.vla_utils import make_att_2d_masks
from lerobot.processor.normalize_processor import NormalizerProcessorStep
from lerobot.utils.constants import OBS_LANGUAGE_ATTENTION_MASK, OBS_LANGUAGE_TOKENS

FLOW_STEPS = policy_cfg.num_inference_steps  # 10（論文と同じ）
CHUNK = policy_cfg.chunk_size  # 50
PAD_DIM = policy_cfg.max_action_dim  # 32（π0.5 内部では行動を 32 次元にゼロ埋め）
ACT_DIM = 7  # LIBERO: 手先の相対位置 3 + 相対回転 3 + グリッパー 1
# 参照行動の大きさ（LIBERO の行動空間 [-1, 1] での値）と、チャンクの先頭何ステップを参照で固定するか。
# 物体が見えている状態で「スープへ向かう」π0.5 を逆向きに誘導できるかで決めた（開発時の比較）:
#   0.5 / 10 ステップ（論文と同じ方式）では逆向き（ロボット基準の +y）へ動かせず、1.0 / チャンク全体で +0.23 m 動いた。
REF_SPEED = 1.0
REF_STEPS = CHUNK

# 指示 -> ロボット基準座標での移動方向。名前は固定カメラの画面上の向きに合わせる
# （ロボット基準の +y は画面の右。固定カメラはロボットの正面からロボットを向いている）
DIRECTIONS = {
    "right": np.array([0.0, 1.0, 0.0]),
    "left": np.array([0.0, -1.0, 0.0]),
    "forward": np.array([1.0, 0.0, 0.0]),  # 手前（カメラ側）
    "back": np.array([-1.0, 0.0, 0.0]),  # 奥（ロボット側）
}
GRIPPER_OPEN = -1.0

normalizer = next(s for s in pre.steps if isinstance(s, NormalizerProcessorStep))


@torch.no_grad()
def velocity_field(batch):
    """観測で条件づけた速度場 v(x, t) を返す。画像・言語部分の KV キャッシュは 1 回だけ計算する。

    LeRobot の PI05Pytorch.sample_actions の前半と同じ処理。
    """
    m = policy.model
    images, img_masks = policy._preprocess_images(batch)
    states, state_masks = policy._prepare_memory_states(batch)
    embs, pad, att = m.embed_prefix(
        images, img_masks, batch[OBS_LANGUAGE_TOKENS], batch[OBS_LANGUAGE_ATTENTION_MASK], states, state_masks
    )
    att_4d = m._prepare_attention_masks_4d(
        make_att_2d_masks(pad, att), dtype=embs.dtype if m.use_typed_attention_masks else None
    )
    m.paligemma_with_expert.paligemma.model.language_model.config._attn_implementation = "eager"
    _, kv = m.paligemma_with_expert.forward(
        attention_mask=att_4d,
        position_ids=torch.cumsum(pad, dim=1) - 1,
        past_key_values=None,
        inputs_embeds=[embs, None],
        use_cache=True,
    )

    def v(x, t):
        time = torch.full((x.shape[0],), t, dtype=torch.float32, device=x.device)
        return m.denoise_step(prefix_pad_masks=pad, past_key_values=kv, x_t=x, timestep=time)

    return v


def integrate_forward(v, noise):
    """ノイズ -> 行動（t: 1 -> 0）。LeRobot の euler_integrate と同じ更新式。"""
    x, dt = noise, -1.0 / FLOW_STEPS
    for k in range(FLOW_STEPS):
        x = x + dt * v(x, 1.0 + k * dt)
    return x


def integrate_reverse(v, action):
    """行動 -> ノイズ（t: 0 -> 1）。順方向のオイラー法を 1 ステップずつ逆にたどる。"""
    x, h = action, 1.0 / FLOW_STEPS
    for k in range(FLOW_STEPS):
        x = x + h * v(x, k * h)
    return x


def make_reference(command, speed=None):
    """指示から参照行動チャンク (1, CHUNK, PAD_DIM) を作る（正規化された行動空間）。"""
    speed = REF_SPEED if speed is None else speed
    raw = torch.zeros(1, CHUNK, ACT_DIM)
    raw[..., :3] = torch.as_tensor(DIRECTIONS[command] * speed, dtype=torch.float32)
    raw[..., 6] = GRIPPER_OPEN
    ref = torch.zeros(1, CHUNK, PAD_DIM, device=DEVICE)
    ref[..., :ACT_DIM] = normalizer._normalize_action(raw, inverse=False).to(DEVICE)
    return ref


@torch.no_grad()
def frs_chunk(batch, command, ref_steps=None, return_noise=False):
    """FRS: 参照行動を逆積分してノイズにし、順積分で π0.5 の行動に戻す。

    論文と同様、先頭 ref_steps 以外のノイズは N(0, I) に置き換え、残りは π0.5 に補完させる。
    return_noise=True なら、π0.5 に入力したノイズも返す（可視化用）。
    """
    ref_steps = REF_STEPS if ref_steps is None else ref_steps
    v = velocity_field(batch)
    noise = integrate_reverse(v, make_reference(command))
    noise[:, ref_steps:] = torch.randn_like(noise[:, ref_steps:])
    actions = integrate_forward(v, noise)[:, :N_ACTION_STEPS, :ACT_DIM]
    return (actions, noise) if return_noise else actions


## セル5: 人間の指示 + FRS で探索し、缶が見えたら π0.5 に引き渡す（動画と同じ指示列を自動で再生）

In [ ]:
# セル5: 人間の指示 + FRS で探索し、缶が見えたら π0.5 に引き渡す（動画と同じ指示列を自動で再生）
# 指示: auto（様子見: π0.5 に任せる）/ right・left・forward・back（FRS で誘導）/ handover（以降はすべて π0.5 に任せる）
# 1 指示 = 10 ステップ。方向は固定カメラの画面上の向き（back = 奥 = ロボット側、forward = 手前）。
COMMANDS = ["auto"] * 2 + ["right"] * 5 + ["back"] * 2 + ["handover"]


class Operation:
    """缶を隠した配置で、人間が 1 指示ずつ操作するセッション（セル6 のボタン操作でも使う）。"""

    def __init__(self, seed=0):
        torch.manual_seed(seed)
        self.obs = reset_hidden(seed)
        self.frames, self.steps, self.success, self.finished = [frame_of(self.obs)], 0, False, False

    def command(self, cmd, on_update=None):
        """1 指示を実行する。handover なら、以降は π0.5 だけで最後まで動かす。"""
        if self.finished:
            return
        if cmd == "handover":
            while not self.finished:
                self._run(pi05_chunk(to_batch(self.obs)))
                if on_update:
                    on_update()
            return
        batch = to_batch(self.obs)
        self._run(frs_chunk(batch, cmd) if cmd in DIRECTIONS else pi05_chunk(batch))
        if on_update:
            on_update()

    def _run(self, chunk):
        for i in range(chunk.shape[1]):
            self.obs, _, terminated, _, _ = vec_env.step(post(chunk[:, i]).cpu().numpy())
            self.frames.append(frame_of(self.obs))
            self.steps += 1
            self.success = bool(env._env.check_success())
            if self.success or terminated[0] or self.steps >= MAX_STEPS:
                self.finished = True
                break

    def report(self):
        if self.success:
            return f"成功: スープ缶をかごに入れた（{self.steps} ステップ）"
        others = [NAMES.get(n, n) for n in objects_in_basket() if n != TARGET]
        return "失敗: " + (f"別の物体（{'、'.join(others)}）をかごに入れた" if others else "スープ缶をかごに入れられなかった")


op = Operation()
for cmd in COMMANDS:
    op.command(cmd)
show_video(op.frames, "frs_search.mp4")
print("指示列:", " → ".join(COMMANDS))
print(op.report())

## セル6: ボタンで操作して試す（缶を隠した配置。1 回押すごとに 10 ステップ進む）

In [ ]:
# セル6: ボタンで操作して試す（缶を隠した配置。1 回押すごとに 10 ステップ進む）
# 様子見で π0.5 の動きを見て、うまくいかなそうなら方向ボタンで誘導し、手先カメラに缶が映ったら「引き渡し」を押す。
# T4 では 1 回の操作に数秒かかる。もう一度試すときはこのセルを再実行する。
import io

import ipywidgets as widgets
from IPython.display import clear_output
from PIL import Image as PILImage

try:  # Colab で ipywidgets を表示するための設定
    from google.colab import output

    output.enable_custom_widget_manager()
except Exception:  # noqa: BLE001  Colab 以外（Jupyter など）ではそのまま表示できる
    pass


def png(frame):
    buf = io.BytesIO()
    PILImage.fromarray(frame).save(buf, format="png")
    return buf.getvalue()


session = Operation()
view = widgets.Image(value=png(session.frames[-1]), format="png", width=720)
status = widgets.Label(value="開始: スープ缶はかごの裏に隠れている（左: 固定カメラ / 右: 手先カメラ）")
result = widgets.Output()
BUTTONS = [("様子見", "auto"), ("↑ 奥", "back"), ("↓ 手前", "forward"), ("← 左", "left"), ("→ 右", "right"), ("引き渡し", "handover")]
buttons = [widgets.Button(description=label, layout=widgets.Layout(width="110px")) for label, _ in BUTTONS]


def update():
    view.value = png(session.frames[-1])
    status.value = f"ステップ {session.steps}"


def on_click(cmd):
    def handler(_):
        for b in buttons:
            b.disabled = True
        status.value = "実行中…"
        session.command(cmd, on_update=update)
        if session.finished:
            with result:
                clear_output()
                show_video(session.frames, "interactive.mp4")
                print(session.report())
        else:
            for b in buttons:
                b.disabled = False

    return handler


for b, (_, cmd) in zip(buttons, BUTTONS):
    b.on_click(on_click(cmd))
display(widgets.VBox([view, status, widgets.HBox(buttons), result]))